# sEMG Gesture Recognition con Lava-DL

Notebook sperimentale per addestrare una rete SNN su segnali sEMG a 8 canali.

Pipeline principale:
1. caricamento CSV etichettati
2. split temporale train/validation/test
3. finestratura con controllo di purezza della label
4. delta encoding positivo/negativo
5. training Lava-DL/SLAYER
6. valutazione e salvataggio dei risultati


In [ ]:
# Standard library
import json
import os
import random
import re
import shutil
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

# Scientific computing and plotting
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision

# Scikit-learn utilities and metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

# Lava-DL / SLAYER
import lava
import lava.lib.dl.slayer as slayer


## 1. Configurazione e caricamento dati

La variabile `DATA_VERSION` seleziona se usare i CSV filtrati (`dataset/`) oppure i raw labellati (`1ks_labeled/`). Ogni file deve contenere le colonne `CH1`-`CH8` e `label`.


In [ ]:
def find_project_root():
    env_root = os.environ.get("PIPELINE_EMG_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if not (root / "pyproject.toml").exists():
            raise FileNotFoundError(f"PIPELINE_EMG_ROOT non valido: {root}")
        return root
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "dataset").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Radice pipeline_emg non trovata. Imposta PIPELINE_EMG_ROOT.")

PROJECT_ROOT = find_project_root()

# Scegli quali dati usare per il training:
# Il pacchetto distribuito include il dataset filtrato e labellato.
DATA_VERSION = "filtered"
SEED = 123
RUN_TEST = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATASET_PATHS = {
    "filtered": PROJECT_ROOT / "dataset" / "DATASET_FILTRATO",
}

if DATA_VERSION not in DATASET_PATHS:
    raise ValueError(f"DATA_VERSION deve essere una tra {list(DATASET_PATHS)}")

DATASET_PATH = DATASET_PATHS[DATA_VERSION]
CHANNEL_COLS = [f"CH{i}" for i in range(1, 9)]
LABEL_COL = "label"

csv_files = [
    path for path in DATASET_PATH.glob("*.csv")
    if "_aug_" not in path.name
]

print(f"Data version: {DATA_VERSION}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Found {len(csv_files)} CSV files in the dataset directory.")

for f in csv_files:
    print(f.name)


In [ ]:
dfs ={}

for f in csv_files:
    df = pd.read_csv(f)
    dfs[f.stem] = df

    print(f"\n=== {f.name} ===")
    print("Shape:", df.shape)
    print("Colonne:", df.columns.tolist())
    #display(df.head())

In [ ]:
all_data = {}

for name, df in dfs.items():
    all_data[name] = {
        "emg": df[CHANNEL_COLS].to_numpy(dtype=np.float32),
        "label": df[LABEL_COL].to_numpy(dtype=np.int64),
    }

for name, data in all_data.items():
    print(
        name,
        "emg:", data["emg"].shape,
        "label:", data["label"].shape,
        "classi:", sorted(np.unique(data["label"]))
    )

In [ ]:
for name, data in all_data.items():
    counts = pd.Series(data["label"]).value_counts().sort_index()

    print(f"\n=== {name} ===")
    print(counts)

    counts.plot(kind="bar", title=f"Distribuzione classi - {name}")
    #plt.xlabel("Classe")
    #plt.ylabel("Campioni")
    #plt.show()

In [ ]:
all_labels = np.concatenate([data["label"] for data in all_data.values()])
classes = sorted(np.unique(all_labels))

print("Classi globali:", classes)
print("Numero classi:", len(classes))

NUM_CLASSES = len(classes)
assert NUM_CLASSES == 4, f"Mi aspettavo 4 classi, trovate {NUM_CLASSES}: {classes}"

In [ ]:
FS = 1000
SKIP_INITIAL_SECONDS = 1
SKIP_INITIAL_SAMPLES = int(FS * SKIP_INITIAL_SECONDS)

dfs = {}

for f in csv_files:
    df = pd.read_csv(f)

    # Elimina l'avvio della registrazione e ricrea l'indice.
    df = df.iloc[SKIP_INITIAL_SAMPLES:].reset_index(drop=True)

    dfs[f.stem] = df

    print(f"\n=== {f.name} ===")
    print("Campioni iniziali rimossi:", SKIP_INITIAL_SAMPLES)
    print("Shape:", df.shape)
    print("Classi:", df[LABEL_COL].value_counts().sort_index().to_dict())

## 2. Split train/validation/test

Sono disponibili due modalità di split:

- `temporal_70_15_15`: divide ogni file lungo il tempo in 70% train, 15% validation, 15% test.
- `filename_groups`: usa file diversi per ogni split: `1k` in train, `10` in validation, `10_10_10` in test.


In [ ]:
SPLIT_MODE = "temporal_70_15_15"
#SPLIT_MODE = "filename_groups"

SPLIT_RATIOS = {
    "train": 0.70,
    "valid": 0.15,
    "test": 0.15,
}

# Usati solo con SPLIT_MODE = "filename_groups".
# L'ordine conta: "10_10_10" viene controllato prima di "10".
FILENAME_SPLIT_PATTERNS = {
    "test": ["S2_test"],
    "train": ["S1_"],
    "valid": ["S2_"],
}
UNMATCHED_FILE_ACTION = "skip"  # "skip" oppure "error"


def empty_like(data):
    return {
        "emg": data["emg"][:0],
        "label": data["label"][:0],
    }


def assign_split_from_filename(name):
    name_lower = name.lower()

    for split_name in ("test", "train", "valid"):
        for pattern in FILENAME_SPLIT_PATTERNS[split_name]:
            if pattern.lower() in name_lower:
                return split_name

    if UNMATCHED_FILE_ACTION == "skip":
        return None

    raise ValueError(
        f"Non so assegnare il file '{name}' con SPLIT_MODE='filename_groups'. "
        "Aggiorna FILENAME_SPLIT_PATTERNS oppure usa UNMATCHED_FILE_ACTION='skip'."
    )


train_data = {}
valid_data = {}
test_data = {}
skipped_files = []

if SPLIT_MODE == "temporal_70_15_15":
    for name, data in all_data.items():
        emg = data["emg"]
        label = data["label"]

        n = len(emg)
        i_train = int(n * SPLIT_RATIOS["train"])
        i_valid = int(n * (SPLIT_RATIOS["train"] + SPLIT_RATIOS["valid"]))

        train_data[name] = {
            "emg": emg[:i_train],
            "label": label[:i_train],
        }

        valid_data[name] = {
            "emg": emg[i_train:i_valid],
            "label": label[i_train:i_valid],
        }

        test_data[name] = {
            "emg": emg[i_valid:],
            "label": label[i_valid:],
        }

elif SPLIT_MODE == "filename_groups":
    for name, data in all_data.items():
        split_name = assign_split_from_filename(name)

        if split_name is None:
            skipped_files.append(name)
        elif split_name == "train":
            train_data[name] = data
        elif split_name == "valid":
            valid_data[name] = data
        elif split_name == "test":
            test_data[name] = data

else:
    raise ValueError("SPLIT_MODE deve essere 'temporal_70_15_15' oppure 'filename_groups'")

print("Split completato:", SPLIT_MODE)
print("train files:", sorted(train_data))
print("valid files:", sorted(valid_data))
print("test files:", sorted(test_data))
if skipped_files:
    print("skipped files:", sorted(skipped_files))

if SPLIT_MODE == "filename_groups":
    missing = [
        split_name
        for split_name, split_data in {
            "train": train_data,
            "valid": valid_data,
            "test": test_data,
        }.items()
        if not split_data
    ]
    if missing:
        raise ValueError(
            f"Split vuoto per: {missing}. "
            "Controlla i nomi dei file o aggiorna FILENAME_SPLIT_PATTERNS."
        )


In [ ]:
def label_counts(labels):
    if len(labels) == 0:
        return {}
    return dict(pd.Series(labels).value_counts().sort_index())


def check_split(split_name, split_data):
    if not split_data:
        print(f"\n=== {split_name} ===")
        print("Nessun file assegnato")
        return

    labels = np.concatenate([data["label"] for data in split_data.values()])
    counts = pd.Series(labels).value_counts().sort_index()

    print(f"\n=== {split_name} ===")
    print("File:", sorted(split_data))
    print("Totale campioni:", len(labels))
    print("Classi:", counts.to_dict())


def check_split_by_file(all_data, train_data, valid_data, test_data):
    rows = []

    for name, data in all_data.items():
        n_total = len(data["label"])
        train_labels = train_data.get(name, empty_like(data))["label"]
        valid_labels = valid_data.get(name, empty_like(data))["label"]
        test_labels = test_data.get(name, empty_like(data))["label"]

        rows.append({
            "file": name,
            "assigned_split": (
                "train" if name in train_data else
                "valid" if name in valid_data else
                "test" if name in test_data else
                "skipped"
            ),
            "total_samples": n_total,
            "train_samples": len(train_labels),
            "valid_samples": len(valid_labels),
            "test_samples": len(test_labels),
            "train_pct": len(train_labels) / n_total,
            "valid_pct": len(valid_labels) / n_total,
            "test_pct": len(test_labels) / n_total,
            "train_classes": label_counts(train_labels),
            "valid_classes": label_counts(valid_labels),
            "test_classes": label_counts(test_labels),
        })

    return pd.DataFrame(rows)


check_split("train", train_data)
check_split("valid", valid_data)
check_split("test", test_data)

split_check_df = check_split_by_file(all_data, train_data, valid_data, test_data)
display(split_check_df)


In [ ]:
# Augmentation applicata esclusivamente al training, dopo lo split.
# Non viene applicata alcuna traslazione temporale.
USE_TRAIN_AUGMENTATION = False
AUGMENTATION_COPIES = 1
GLOBAL_GAIN_RANGE = (0.90, 1.10)
CHANNEL_GAIN_RANGE = (0.90, 1.10)
NOISE_STD_FRACTION_RANGE = (0.0, 0.0)
CROSSTALK_RANGE = (0.0, 0.0)
ADC_MIN = -(2**23)
ADC_MAX = 2**23 - 1


def augment_emg_recording(emg, rng):
    x = np.asarray(emg, dtype=np.float64).copy()

    # Variazione dell'intensita complessiva della contrazione.
    x *= rng.uniform(*GLOBAL_GAIN_RANGE)

    # Differenze di guadagno/contatto indipendenti per ciascun canale.
    channel_gains = rng.uniform(*CHANNEL_GAIN_RANGE, size=(1, x.shape[1]))
    x *= channel_gains

    # Crosstalk leggero solo tra canali adiacenti.
    source = x.copy()
    for channel in range(x.shape[1]):
        if channel > 0:
            x[:, channel] += rng.uniform(*CROSSTALK_RANGE) * source[:, channel - 1]
        if channel + 1 < x.shape[1]:
            x[:, channel] += rng.uniform(*CROSSTALK_RANGE) * source[:, channel + 1]

    # Rumore proporzionale alla deviazione standard di ogni canale.
    channel_std = np.std(x, axis=0, keepdims=True)
    noise_fraction = rng.uniform(
        *NOISE_STD_FRACTION_RANGE,
        size=(1, x.shape[1]),
    )
    x += rng.normal(size=x.shape) * channel_std * noise_fraction

    # Quantizzazione finale: i valori aumentati restano conteggi ADC interi.
    return np.clip(np.rint(x), ADC_MIN, ADC_MAX).astype(np.int32)


def build_augmented_training_split(original_train_data, seed):
    augmented = {
        name: {"emg": data["emg"], "label": data["label"]}
        for name, data in original_train_data.items()
    }
    rng = np.random.default_rng(seed)

    for name in sorted(original_train_data):
        data = original_train_data[name]
        for copy_idx in range(1, AUGMENTATION_COPIES + 1):
            augmented[f"{name}_notebook_aug_{copy_idx:02d}"] = {
                "emg": augment_emg_recording(data["emg"], rng),
                "label": data["label"].copy(),
            }

    return augmented


if USE_TRAIN_AUGMENTATION:
    train_data_for_windows = build_augmented_training_split(train_data, SEED + 10_000)
else:
    train_data_for_windows = train_data

augmented_names = [name for name in train_data_for_windows if "_notebook_aug_" in name]
print("Augmentation training:", "ON" if USE_TRAIN_AUGMENTATION else "OFF")
print("Registrazioni originali train:", len(train_data))
print("Registrazioni aumentate:", len(augmented_names))
if augmented_names:
    print("Dtype augmentation:", train_data_for_windows[augmented_names[0]]["emg"].dtype)


## 3. Finestratura e assegnazione label

Ogni finestra riceve la label dominante. Le finestre troppo ambigue possono essere scartate usando min_label_ratio.


In [ ]:
def majority_label_with_confidence(labels, min_ratio=0.85):
    values, counts = np.unique(labels, return_counts=True)
    idx = np.argmax(counts)

    winner = values[idx]
    ratio = counts[idx] / len(labels)

    if ratio < min_ratio:
        return None

    return winner


def create_windows(split_data, window_size=250, window_shift=50, min_label_ratio=0.85):
    x = []
    y = []
    skipped = 0

    for name, data in split_data.items():
        emg = data["emg"]
        label = data["label"]

        start = 0
        while start + window_size <= len(emg):
            end = start + window_size

            win_emg = emg[start:end]
            win_label = label[start:end]

            label_out = majority_label_with_confidence(
                win_label,
                min_ratio=min_label_ratio
            )

            if label_out is None:
                skipped += 1
                start += window_shift
                continue

            x.append(win_emg)
            y.append(label_out)

            start += window_shift

    x = np.array(x, dtype=np.float32)
    y = np.array(y, dtype=np.int64)

    print(f"Finestre scartate: {skipped}")

    return x, y

In [ ]:
WINDOW_SIZE = 300 # numero di campioni per finestra
WINDOW_SHIFT = 25   # shift in campioni tra finestre consecutive
MIN_LABEL_RATIO = 0.9


x_train_raw, y_train = create_windows(train_data_for_windows, WINDOW_SIZE, WINDOW_SHIFT)
x_valid_raw, y_valid = create_windows(valid_data, WINDOW_SIZE, WINDOW_SHIFT)
x_test_raw, y_test = create_windows(test_data, WINDOW_SIZE, WINDOW_SHIFT)

print("x_train_raw:", x_train_raw.shape, "y_train:", y_train.shape)
print("x_valid_raw:", x_valid_raw.shape, "y_valid:", y_valid.shape)
print("x_test_raw:", x_test_raw.shape, "y_test:", y_test.shape)

In [ ]:
for name, y in [
    ("train", y_train),
    ("valid", y_valid),
    ("test", y_test),
]:
    print(f"\n=== {name} ===")
    print(pd.Series(y).value_counts().sort_index())

## 4. Analisi del segnale e scelta del delta

Queste celle aiutano a capire scala dei valori, differenze campione-campione e spike rate atteso per diversi valori di DELTA.


In [ ]:
def delta_encode_window(window, delta=50.0):
    """
    window shape: (time, channels)
    output shape: (channels * 2, time)
    """
    time_steps, channels = window.shape
    spikes = np.zeros((channels * 2, time_steps), dtype=np.float32)

    baseline = window[0].copy()

    for t in range(time_steps):
        diff = window[t] - baseline

        pos = diff > delta
        neg = diff < -delta

        spikes[0::2, t] = pos.astype(np.float32)
        spikes[1::2, t] = neg.astype(np.float32)

        baseline[pos | neg] = window[t, pos | neg]

    return spikes

In [ ]:
all_emg = np.concatenate([data["emg"] for data in all_data.values()], axis=0)

print("Shape:", all_emg.shape)
print("min:", np.min(all_emg))
print("max:", np.max(all_emg))
print("mean:", np.mean(all_emg))
print("std:", np.std(all_emg))

percentiles = [0, 0.1, 1, 5, 25, 50, 75, 95, 99, 99.9, 100]
for p in percentiles:
    print(f"p{p}: {np.percentile(all_emg, p):.3f}")

In [ ]:
stats_per_channel = pd.DataFrame({
    "channel": CHANNEL_COLS,
    "min": all_emg.min(axis=0),
    "max": all_emg.max(axis=0),
    "mean": all_emg.mean(axis=0),
    "std": all_emg.std(axis=0),
    "p1": np.percentile(all_emg, 1, axis=0),
    "p50": np.percentile(all_emg, 50, axis=0),
    "p99": np.percentile(all_emg, 99, axis=0),
})

display(stats_per_channel)

In [ ]:
all_diffs = []

for data in all_data.values():
    emg = data["emg"]
    diff = np.diff(emg, axis=0)
    all_diffs.append(diff)

all_diffs = np.concatenate(all_diffs, axis=0)
abs_diffs = np.abs(all_diffs)

print("diff shape:", all_diffs.shape)
print("abs diff mean:", abs_diffs.mean())
print("abs diff std:", abs_diffs.std())
print("abs diff max:", abs_diffs.max())

percentiles = [50, 75, 90, 95, 97, 99, 99.5, 99.9]
for p in percentiles:
    print(f"abs diff p{p}: {np.percentile(abs_diffs, p):.3f}")

In [ ]:
diff_stats_per_channel = pd.DataFrame({
    "channel": CHANNEL_COLS,
    "abs_diff_mean": abs_diffs.mean(axis=0),
    "abs_diff_std": abs_diffs.std(axis=0),
    "abs_diff_p90": np.percentile(abs_diffs, 90, axis=0),
    "abs_diff_p95": np.percentile(abs_diffs, 95, axis=0),
    "abs_diff_p99": np.percentile(abs_diffs, 99, axis=0),
    "abs_diff_max": abs_diffs.max(axis=0),
})

display(diff_stats_per_channel)

In [ ]:
def estimate_delta_spike_rate(split_data, deltas):
    rows = []

    for delta in deltas:
        total_spikes = 0
        total_slots = 0

        for data in split_data.values():
            emg = data["emg"]
            baseline = emg[0].copy()

            spikes = 0
            slots = emg.shape[0] * emg.shape[1] * 2

            for t in range(emg.shape[0]):
                diff = emg[t] - baseline
                pos = diff > delta
                neg = diff < -delta

                spikes += pos.sum() + neg.sum()
                baseline[pos | neg] = emg[t, pos | neg]

            total_spikes += spikes
            total_slots += slots

        rows.append({
            "delta": delta,
            "spike_rate": total_spikes / total_slots,
            "total_spikes": total_spikes,
        })

    return pd.DataFrame(rows)

candidate_deltas = [10, 25, 50, 100, 200, 500, 1000, 2000, 5000]

delta_stats = estimate_delta_spike_rate(train_data_for_windows, candidate_deltas)
display(delta_stats)

In [ ]:
# Parametri principali dell'encoding.
# ENCODING_MODE:
# - "standard": delta encoding classico, il riferimento salta al valore corrente quando spika
# - "lambda": delta encoding con riferimento che decade secondo lambda_d e si muove di +/- DELTA
#
# MAX_DERIVATIVE_ORDER:
# - 0: solo segnale originale
# - 1: segnale originale + derivata prima
# - 2: segnale originale + derivata prima + derivata seconda
DELTA = 200
ENCODING_MODE = "standard"  # cambia in "lambda" per provare raw + lambda encoding
LAMBDA_D = 0.0001
MAX_DERIVATIVE_ORDER = 2

# Soglia delta per ordine: [segnale originale, derivata prima, derivata seconda]
# Se le derivate spikano troppo, aumenta i valori successivi, ad esempio [500, 1000, 1500].
DELTA_BY_ORDER = [DELTA, DELTA, DELTA]

print("DATA_VERSION:", DATA_VERSION)
print("ENCODING_MODE:", ENCODING_MODE)
print("DELTA:", DELTA)
print("LAMBDA_D:", LAMBDA_D)
print("MAX_DERIVATIVE_ORDER:", MAX_DERIVATIVE_ORDER)
print("DELTA_BY_ORDER:", DELTA_BY_ORDER[:MAX_DERIVATIVE_ORDER + 1])


## 5. Delta encoding

La variabile `ENCODING_MODE` seleziona l’encoder usato per il training: `standard` oppure `lambda`. `MAX_DERIVATIVE_ORDER` permette di aggiungere derivata prima e seconda prima dell’encoding.


In [ ]:
def expand_window_with_derivatives(window, max_derivative_order=0):
    """
    Espande una finestra aggiungendo derivate discrete come nell'altro notebook.

    Input:
        window shape: (time, channels)

    Output:
        expanded_window shape: (time_trimmed, channels * (max_derivative_order + 1))

    Ordine feature:
        [CH originali, CH derivata prima, CH derivata seconda, ...]
    """
    window = np.asarray(window, dtype=np.float32)

    if max_derivative_order == 0:
        return window

    time_steps, old_dim_size = window.shape
    new_dim_size = old_dim_size * (max_derivative_order + 1)

    expanded = np.zeros((time_steps, new_dim_size), dtype=np.float32)
    expanded[:, :old_dim_size] = window

    for n in range(1, max_derivative_order + 1):
        prev_start = old_dim_size * (n - 1)
        curr_start = old_dim_size * n

        for ch in range(old_dim_size):
            for j in range(time_steps - n * 4):
                expanded[j + n * 2, curr_start + ch] = (
                    -expanded[j, prev_start + ch]
                    -2 * expanded[j + 1, prev_start + ch]
                    +2 * expanded[j + 2, prev_start + ch]
                    +expanded[j + 3, prev_start + ch]
                )

    trim = max_derivative_order * 2
    return expanded[trim:-trim, :]


def make_delta_vector(n_features, original_channels, delta_by_order):
    """
    Crea una soglia delta per ogni feature espansa.
    Le prime original_channels feature usano delta_by_order[0],
    le successive original_channels usano delta_by_order[1], ecc.
    """
    delta_values = np.zeros(n_features, dtype=np.float32)

    for feature_idx in range(n_features):
        derivative_order = feature_idx // original_channels
        delta_values[feature_idx] = delta_by_order[derivative_order]

    return delta_values


def delta_encode_window_standard(expanded_window, delta_values):
    """
    Delta encoding standard su una finestra già espansa.

    Input:
        expanded_window shape: (time, features)
    Output:
        spikes shape: (features * 2, time)
    """
    time_steps, features = expanded_window.shape
    spikes = np.zeros((features * 2, time_steps), dtype=np.float32)

    ref = expanded_window[0].copy()

    for t in range(time_steps):
        diff = expanded_window[t] - ref

        pos = diff > delta_values
        neg = diff < -delta_values

        spikes[0::2, t] = pos.astype(np.float32)
        spikes[1::2, t] = neg.astype(np.float32)

        fired = pos | neg
        ref[fired] = expanded_window[t, fired]

    return spikes


def delta_encode_window_lambda(expanded_window, delta_values, lambda_d=0.001):
    """
    Delta encoding con lambda su una finestra già espansa.
    """
    time_steps, features = expanded_window.shape
    spikes = np.zeros((features * 2, time_steps), dtype=np.float32)

    ref = expanded_window[0].copy()

    for t in range(1, time_steps):
        ref = ref * (1.0 - lambda_d)

        diff = expanded_window[t] - ref

        pos = diff >= delta_values
        neg = diff <= -delta_values

        for feature_idx in range(features):
            if pos[feature_idx]:
                spikes[2 * feature_idx, t] = 1.0
                ref[feature_idx] = ref[feature_idx] + delta_values[feature_idx]

            if neg[feature_idx]:
                spikes[2 * feature_idx + 1, t] = 1.0
                ref[feature_idx] = ref[feature_idx] - delta_values[feature_idx]

    return spikes


def delta_encode_window(
    window,
    delta_by_order,
    mode="standard",
    lambda_d=0.001,
    max_derivative_order=0,
):
    original_channels = window.shape[1]
    expanded_window = expand_window_with_derivatives(
        window,
        max_derivative_order=max_derivative_order,
    )
    delta_values = make_delta_vector(
        n_features=expanded_window.shape[1],
        original_channels=original_channels,
        delta_by_order=delta_by_order,
    )

    if mode == "standard":
        return delta_encode_window_standard(expanded_window, delta_values)

    if mode == "lambda":
        return delta_encode_window_lambda(
            expanded_window,
            delta_values,
            lambda_d=lambda_d,
        )

    raise ValueError("mode deve essere 'standard' oppure 'lambda'")


def delta_encode_dataset(
    x_raw,
    delta_by_order,
    mode="standard",
    lambda_d=0.001,
    max_derivative_order=0,
):
    if max_derivative_order < 0:
        raise ValueError("max_derivative_order deve essere >= 0")

    if len(delta_by_order) < max_derivative_order + 1:
        raise ValueError(
            "delta_by_order deve contenere almeno max_derivative_order + 1 valori"
        )

    return np.stack([
        delta_encode_window(
            window,
            delta_by_order=delta_by_order,
            mode=mode,
            lambda_d=lambda_d,
            max_derivative_order=max_derivative_order,
        )
        for window in x_raw
    ]).astype(np.float32)


In [ ]:
x_train = delta_encode_dataset(
    x_train_raw,
    DELTA_BY_ORDER,
    mode=ENCODING_MODE,
    lambda_d=LAMBDA_D,
    max_derivative_order=MAX_DERIVATIVE_ORDER,
)
x_valid = delta_encode_dataset(
    x_valid_raw,
    DELTA_BY_ORDER,
    mode=ENCODING_MODE,
    lambda_d=LAMBDA_D,
    max_derivative_order=MAX_DERIVATIVE_ORDER,
)
x_test = delta_encode_dataset(
    x_test_raw,
    DELTA_BY_ORDER,
    mode=ENCODING_MODE,
    lambda_d=LAMBDA_D,
    max_derivative_order=MAX_DERIVATIVE_ORDER,
)

print("Data version:", DATA_VERSION)
print("Encoding mode:", ENCODING_MODE)
print("Delta by order:", DELTA_BY_ORDER[:MAX_DERIVATIVE_ORDER + 1])
print("Lambda_d:", LAMBDA_D)
print("Max derivative order:", MAX_DERIVATIVE_ORDER)
print("x_train:", x_train.shape)
print("x_valid:", x_valid.shape)
print("x_test:", x_test.shape)


In [ ]:
def spike_stats(name, x):
    spikes_per_window = x.sum(axis=(1, 2))

    print(f"\n=== {name} ===")
    print("shape:", x.shape)
    print("spike rate:", x.mean())
    print("spike/window mean:", spikes_per_window.mean())
    print("spike/window min:", spikes_per_window.min())
    print("spike/window max:", spikes_per_window.max())

spike_stats("train", x_train)
spike_stats("valid", x_valid)
spike_stats("test", x_test)

In [ ]:
WINDOW_IDX = 320
CHANNEL_IDX = 5

raw_window = x_train_raw[WINDOW_IDX]
encoded_window = x_train[WINDOW_IDX]

trim = MAX_DERIVATIVE_ORDER * 2

if trim > 0:
    signal = raw_window[trim:-trim, CHANNEL_IDX]
else:
    signal = raw_window[:, CHANNEL_IDX]

t = np.arange(len(signal))

pos_spikes = encoded_window[CHANNEL_IDX * 2]
neg_spikes = encoded_window[CHANNEL_IDX * 2 + 1]

plt.figure(figsize=(16, 5))

plt.plot(
    t,
    signal,
    label=f"{CHANNEL_COLS[CHANNEL_IDX]} raw",
    color="black",
    linewidth=1,
)

plt.scatter(
    t[pos_spikes == 1],
    signal[pos_spikes == 1],
    color="red",
    s=30,
    label="positive spike",
    zorder=3,
)

plt.scatter(
    t[neg_spikes == 1],
    signal[neg_spikes == 1],
    color="blue",
    s=30,
    label="negative spike",
    zorder=3,
)

plt.title(
    f"Delta encoding - window {WINDOW_IDX}, {CHANNEL_COLS[CHANNEL_IDX]}, "
    f"label={y_train[WINDOW_IDX]}, delta={DELTA}, derivative_order={MAX_DERIVATIVE_ORDER}"
)
plt.xlabel("Time step")
plt.ylabel("Amplitude")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Dataset PyTorch e DataLoader

Wrapper PyTorch per passare finestre spike e label alla rete SNN.


In [ ]:
class EMGDataset(Dataset):
    def __init__(self, x, y):
        self.x = x.astype(np.float32)
        self.y = y.astype(np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.x[idx], dtype=torch.float32),
            torch.tensor(self.y[idx], dtype=torch.long),
        )


BATCH_SIZE = 16

train_dataset = EMGDataset(x_train, y_train)
valid_dataset = EMGDataset(x_valid, y_valid)
test_dataset = EMGDataset(x_test, y_test)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
)
valid_dataloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(len(train_dataset), len(valid_dataset), len(test_dataset))
print(x_train.shape, y_train.shape)

## 7. Modello SNN

Rete dense CUBA in Lava-DL/SLAYER. L'ultimo layer produce le 4 classi: rest, sasso, forbici, carta.


In [ ]:
NUM_CLASSES = 4
INPUT_FEATURES = x_train.shape[1]

class Network(torch.nn.Module):
    def __init__(self):
        super(Network, self).__init__()

        neuron_params = {
            "threshold": 1.25,
            "current_decay": 1,
            "voltage_decay": 0.03,
            "tau_grad": 0.03,
            "scale_grad": 3,
            "requires_grad": True,
        }

        neuron_params_drop = {
            **neuron_params,
            "dropout": slayer.neuron.Dropout(p=0.05),
        }

        self.blocks = torch.nn.ModuleList([
            slayer.block.cuba.Dense(neuron_params_drop, INPUT_FEATURES, 64, weight_norm=True, delay=True),
            slayer.block.cuba.Dense(neuron_params_drop, 64, 128, weight_norm=True, delay=True),
            slayer.block.cuba.Dense(neuron_params, 128, NUM_CLASSES, weight_norm=True),
        ])

    def forward(self, spike):
        for block in self.blocks:
            spike = block(spike)
        return spike

    def grad_flow(self, path):
        grad = [b.synapse.grad_norm for b in self.blocks if hasattr(b, "synapse")]

        plt.figure()
        plt.semilogy(grad)
        plt.savefig(path + "gradFlow.png")
        plt.close()

        return grad

    def export_hdf5(self, filename):
        h = h5py.File(filename, "w")
        layer = h.create_group("layer")
        for i, b in enumerate(self.blocks):
            b.export_hdf5(layer.create_group(f"{i}"))

## 8. Setup training

Inizializzazione di modello, scelta della loss, ottimizzatore, statistiche e assistant SLAYER.

`LOSS_TYPE = "spike_rate"` mantiene la loss originale SLAYER.
`LOSS_TYPE = "weighted_spike_rate"` usa la rate loss pesata per classe.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = Network().to(device)

# Scegli qui quale loss usare:
# - "spike_rate": loss originale SLAYER, come prima
# - "weighted_spike_rate": stessa idea di rate loss, ma pesata per classe
LOSS_TYPE = "spike_rate"

TRUE_RATE = 0.15
FALSE_RATE = 0.03

# Ordine classi nel dataset: 0=rest, 1=sasso, 2=forbici, 3=carta.
# Aumenta il peso di sasso se vuoi penalizzare di più gli errori su quella classe.
CLASS_WEIGHTS = torch.tensor([0.7, 1.0, 1.0, 1.5], dtype=torch.float32)


class WeightedSpikeRateLoss(nn.Module):
    def __init__(
        self,
        true_rate: float = 0.15,
        false_rate: float = 0.03,
        class_weights: torch.Tensor | None = None,
        time_first: bool = False,
        reduction: str = "sum",
    ) -> None:
        super().__init__()
        self.true_rate = float(true_rate)
        self.false_rate = float(false_rate)
        self.time_first = bool(time_first)
        self.reduction = reduction

        if class_weights is not None:
            self.register_buffer("class_weights", class_weights.float())
        else:
            self.class_weights = None

    def forward(self, spike_output, label):
        # Nel training SLAYER l'output del modello e' [B, C, T].
        # Se in altri esperimenti usi output [T, B, C], imposta time_first=True.
        if self.time_first:
            spike_rate = spike_output.mean(dim=0)   # [T, B, C] -> [B, C]
        else:
            spike_rate = spike_output.mean(dim=-1)  # [B, C, T] -> [B, C]

        label = label.long()
        one_hot = F.one_hot(label, num_classes=spike_rate.shape[1]).float()
        target = self.true_rate * one_hot + self.false_rate * (1.0 - one_hot)

        # Errore MSE per finestra/campione del batch.
        mse_per_sample = ((spike_rate - target) ** 2).mean(dim=1)

        if self.class_weights is not None:
            mse_per_sample = mse_per_sample * self.class_weights[label]

        if self.reduction == "sum":
            return mse_per_sample.sum()
        if self.reduction == "mean":
            return mse_per_sample.mean()
        if self.reduction == "none":
            return mse_per_sample
        raise ValueError(f"Unsupported reduction: {self.reduction}")


if LOSS_TYPE == "spike_rate":
    error = slayer.loss.SpikeRate(
        true_rate=TRUE_RATE,
        false_rate=FALSE_RATE,
        reduction="sum",
    ).to(device)
elif LOSS_TYPE == "weighted_spike_rate":
    error = WeightedSpikeRateLoss(
        true_rate=TRUE_RATE,
        false_rate=FALSE_RATE,
        class_weights=CLASS_WEIGHTS,
        time_first=False,
        reduction="sum",
    ).to(device)
else:
    raise ValueError(f"Unknown LOSS_TYPE: {LOSS_TYPE}")

print("Loss:", LOSS_TYPE)
if LOSS_TYPE == "weighted_spike_rate":
    print("Class weights [rest, sasso, forbici, carta]:", CLASS_WEIGHTS.tolist())

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=4,
    threshold=1e-4,
    cooldown=1,
    min_lr=1e-5,
)

stats = slayer.utils.LearningStats()

assistant = slayer.utils.Assistant(
    model,
    error,
    optimizer,
    stats,
    classifier=slayer.classifier.Rate.predict,
)


## 9. Training con early stopping

Il training salva in memoria il miglior modello rispetto alla validation loss e lo ripristina al termine.


In [ ]:
EPOCHS = 100
PATIENCE = 10

best_valid_loss = float("inf")
best_epoch = -1
epochs_without_improvement = 0

best_model_state = None

train_loss_hist = []
valid_loss_hist = []
train_acc_hist = []
valid_acc_hist = []

for epoch in range(EPOCHS):
    for input, label in train_dataloader:
        input = input.to(device)
        label = label.to(device)

        assistant.train(input, label)

    for input, label in valid_dataloader:
        input = input.to(device)
        label = label.to(device)

        assistant.test(input, label)

    train_loss = stats.training.loss
    valid_loss = stats.testing.loss
    train_acc = stats.training.accuracy
    valid_acc = stats.testing.accuracy

    train_loss_hist.append(train_loss)
    valid_loss_hist.append(valid_loss)
    train_acc_hist.append(train_acc)
    valid_acc_hist.append(valid_acc)

    print(
        f"Epoch {epoch:03d} | "
        f"train loss {train_loss:.4f} | "
        f"valid loss {valid_loss:.4f} | "
        f"train acc {train_acc:.4f} | "
        f"valid acc {valid_acc:.4f}"
    )

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

        print(f"  best model updated at epoch {epoch}")
    else:
        epochs_without_improvement += 1
        print(f"  no improvement: {epochs_without_improvement}/{PATIENCE}")

    scheduler.step(valid_loss)
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"  learning rate: {current_lr:.2e}")

    stats.update()

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model.to(device)
    print(f"Restored best model from epoch {best_epoch}")

## 10. Valutazione su valid set

Calcolo delle predizioni sul valid set, confusion matrix e classification report.


In [ ]:
# Valutazione su validation set

model.eval()

y_valid_true = []
y_valid_pred = []
valid_scores = []

with torch.no_grad():
    for input, label in valid_dataloader:
        input = input.to(device)
        label = label.to(device)

        output = model(input)

        pred = slayer.classifier.Rate.predict(output)

        scores_batch = output.sum(dim=2)

        y_valid_true.extend(label.cpu().numpy())
        y_valid_pred.extend(pred.cpu().numpy())
        valid_scores.append(scores_batch.cpu().numpy())

y_valid_true = np.array(y_valid_true)
y_valid_pred = np.array(y_valid_pred)
valid_scores = np.concatenate(valid_scores, axis=0)

print("Validation accuracy:", accuracy_score(y_valid_true, y_valid_pred))

labels = [0, 1, 2, 3]
class_names = ["rest", "sasso", "forbici", "carta"]

cm_valid = confusion_matrix(y_valid_true, y_valid_pred, labels=labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_valid,
    display_labels=class_names,
)

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("Confusion Matrix - Validation")
plt.show()

print(
    classification_report(
        y_valid_true,
        y_valid_pred,
        labels=labels,
        target_names=class_names,
        digits=4,
        zero_division=0,
    )
)

## 11. Salvataggio dati

In [ ]:
# Salvataggio risultati training + validation

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

run_name = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = RESULTS_DIR / run_name
run_dir.mkdir(exist_ok=True)

labels = [0, 1, 2, 3]
class_names = ["rest", "sasso", "forbici", "carta"]

metrics_valid = {
    "valid_accuracy": float(accuracy_score(y_valid_true, y_valid_pred)),
    "valid_macro_f1": float(f1_score(y_valid_true, y_valid_pred, average="macro", zero_division=0)),
    "valid_weighted_f1": float(f1_score(y_valid_true, y_valid_pred, average="weighted", zero_division=0)),
    "valid_macro_precision": float(precision_score(y_valid_true, y_valid_pred, average="macro", zero_division=0)),
    "valid_macro_recall": float(recall_score(y_valid_true, y_valid_pred, average="macro", zero_division=0)),
    "valid_per_class_precision": {
        class_names[i]: float(v)
        for i, v in enumerate(
            precision_score(
                y_valid_true,
                y_valid_pred,
                labels=labels,
                average=None,
                zero_division=0,
            )
        )
    },
    "valid_per_class_recall": {
        class_names[i]: float(v)
        for i, v in enumerate(
            recall_score(
                y_valid_true,
                y_valid_pred,
                labels=labels,
                average=None,
                zero_division=0,
            )
        )
    },
    "valid_per_class_f1": {
        class_names[i]: float(v)
        for i, v in enumerate(
            f1_score(
                y_valid_true,
                y_valid_pred,
                labels=labels,
                average=None,
                zero_division=0,
            )
        )
    },
}

config = {
    "seed": int(SEED),
    "data_version": DATA_VERSION,
    "split_mode": SPLIT_MODE,
    "filename_split_patterns": FILENAME_SPLIT_PATTERNS,
    "split_ratios": SPLIT_RATIOS,
    "dataset_path": str(DATASET_PATH),
    "class_names": class_names,
    "augmentation": {
        "enabled": bool(USE_TRAIN_AUGMENTATION),
        "copies": int(AUGMENTATION_COPIES),
        "global_gain_range": list(GLOBAL_GAIN_RANGE),
        "channel_gain_range": list(CHANNEL_GAIN_RANGE),
        "noise_std_fraction_range": list(NOISE_STD_FRACTION_RANGE),
        "crosstalk_range": list(CROSSTALK_RANGE),
        "temporal_shift": False,
        "output_dtype": "int32",
    },
    "encoding_mode": ENCODING_MODE,
    "delta": float(DELTA),
    "lambda_d": float(LAMBDA_D),
    "window_size": int(WINDOW_SIZE),
    "window_shift": int(WINDOW_SHIFT),
    "min_label_ratio": float(MIN_LABEL_RATIO),
    "batch_size": int(BATCH_SIZE),
    "epochs": int(EPOCHS),
    "patience": int(PATIENCE),
    "best_epoch": int(best_epoch),
    "best_valid_loss": float(best_valid_loss),
    "loss_type": LOSS_TYPE,
    "true_rate": float(TRUE_RATE),
    "false_rate": float(FALSE_RATE),
    "class_weights": CLASS_WEIGHTS.tolist() if LOSS_TYPE == "weighted_spike_rate" else None,
    "num_classes": int(NUM_CLASSES),
    "input_features": int(INPUT_FEATURES),
    "network": "Dense 64-128-4",
}

history = {
    "train_loss": [float(x) for x in train_loss_hist],
    "valid_loss": [float(x) for x in valid_loss_hist],
    "train_acc": [float(x) for x in train_acc_hist],
    "valid_acc": [float(x) for x in valid_acc_hist],
}

valid_results_df = pd.DataFrame({
    "y_true": y_valid_true,
    "y_pred": y_valid_pred,
    "correct": y_valid_true == y_valid_pred,
})

valid_results_df["max_score"] = valid_scores.max(axis=1)
valid_results_df["pred_score"] = valid_scores[
    np.arange(len(y_valid_pred)),
    y_valid_pred,
]

sorted_scores = np.sort(valid_scores, axis=1)
valid_results_df["second_score"] = sorted_scores[:, -2]
valid_results_df["margin"] = (
    valid_results_df["max_score"] -
    valid_results_df["second_score"]
)

for i, class_name in enumerate(class_names):
    valid_results_df[f"score_{class_name}"] = valid_scores[:, i]

report_valid_text = classification_report(
    y_valid_true,
    y_valid_pred,
    labels=labels,
    target_names=class_names,
    digits=4,
    zero_division=0,
)

with open(run_dir / "config.json", "w") as f:
    json.dump(config, f, indent=4)

with open(run_dir / "valid_metrics.json", "w") as f:
    json.dump(metrics_valid, f, indent=4)

with open(run_dir / "history.json", "w") as f:
    json.dump(history, f, indent=4)

with open(run_dir / "valid_classification_report.txt", "w") as f:
    f.write(report_valid_text)

valid_results_df.to_csv(run_dir / "valid_predictions.csv", index=False)

np.save(run_dir / "valid_confusion_matrix.npy", cm_valid)
np.save(run_dir / "y_valid_true.npy", y_valid_true)
np.save(run_dir / "y_valid_pred.npy", y_valid_pred)
np.save(run_dir / "valid_scores.npy", valid_scores)

torch.save(model.state_dict(), run_dir / "model_state_dict.pt")
torch.save(model, run_dir / "model_best.pt")

fig, ax = plt.subplots(figsize=(6, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_valid,
    display_labels=class_names,
)
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("Confusion Matrix - Validation")
plt.savefig(run_dir / "valid_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

print("Risultati validation salvati in:", run_dir)
print("\nValidation metrics:")
print(json.dumps(metrics_valid, indent=4))
print("\nValidation classification report:")
print(report_valid_text)

In [ ]:
# ============================================================
# Diagnostica completa: gesti classificati come REST
# Eseguire dopo la valutazione della validation
# ============================================================

FS_DIAGNOSTIC = 1000
MIN_LABEL_RATIO_USED = 0.85
MAX_WINDOWS_TO_PLOT = 6

CLASS_NAMES = {
    0: "rest",
    1: "sasso",
    2: "forbici",
    3: "carta",
}

# ------------------------------------------------------------
# 1. Controllo delle variabili
# ------------------------------------------------------------

required_variables = [
    "valid_data",
    "x_valid_raw",
    "x_valid",
    "y_valid_true",
    "y_valid_pred",
    "valid_scores",
    "WINDOW_SIZE",
    "WINDOW_SHIFT",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Esegui prima caricamento, finestratura, encoding e validation. "
        f"Variabili mancanti: {missing_variables}"
    )

y_true_diag = np.asarray(y_valid_true, dtype=np.int64)
y_pred_diag = np.asarray(y_valid_pred, dtype=np.int64)
scores_diag = np.asarray(valid_scores)
x_valid_diag = np.asarray(x_valid)
x_valid_raw_diag = np.asarray(x_valid_raw)

print("Finestre validation:", len(y_true_diag))
print("Soglia label realmente usata:", MIN_LABEL_RATIO_USED)

# ------------------------------------------------------------
# 2. Ricostruzione dei metadati delle finestre
# ------------------------------------------------------------

metadata = []

for file_name, data in valid_data.items():
    emg = np.asarray(data["emg"])
    labels = np.asarray(data["label"])

    start = 0

    while start + WINDOW_SIZE <= len(emg):
        end = start + WINDOW_SIZE
        window_labels = labels[start:end]

        values, counts = np.unique(
            window_labels,
            return_counts=True,
        )

        winner_index = int(np.argmax(counts))
        winner = int(values[winner_index])
        label_ratio = float(
            counts[winner_index] / len(window_labels)
        )

        if label_ratio >= MIN_LABEL_RATIO_USED:
            metadata.append({
                "file": file_name,
                "start_sample": start,
                "end_sample": end,
                "start_s": start / FS_DIAGNOSTIC,
                "end_s": end / FS_DIAGNOSTIC,
                "label_ratio": label_ratio,
                "window_label": winner,
            })

        start += WINDOW_SHIFT

valid_diagnostic = pd.DataFrame(metadata)

print("Metadati ricostruiti:", len(valid_diagnostic))

if len(valid_diagnostic) != len(y_true_diag):
    raise RuntimeError(
        "Il numero di metadati non coincide con le finestre di validation: "
        f"{len(valid_diagnostic)} != {len(y_true_diag)}. "
        "Verifica che FS, WINDOW_SIZE, WINDOW_SHIFT e la soglia usata "
        "siano gli stessi del training."
    )

# Verifica aggiuntiva dell'ordine delle finestre.
reconstructed_labels = valid_diagnostic[
    "window_label"
].to_numpy(dtype=np.int64)

if not np.array_equal(reconstructed_labels, y_true_diag):
    mismatch_count = int(
        np.sum(reconstructed_labels != y_true_diag)
    )

    raise RuntimeError(
        "I metadati hanno la lunghezza corretta, ma le label non "
        f"corrispondono in {mismatch_count} finestre. "
        "Probabilmente valid_data è stato modificato dopo la finestratura."
    )

# ------------------------------------------------------------
# 3. Metriche per ogni finestra
# ------------------------------------------------------------

valid_diagnostic["true"] = y_true_diag
valid_diagnostic["pred"] = y_pred_diag

valid_diagnostic["true_name"] = [
    CLASS_NAMES.get(int(value), str(value))
    for value in y_true_diag
]

valid_diagnostic["pred_name"] = [
    CLASS_NAMES.get(int(value), str(value))
    for value in y_pred_diag
]

valid_diagnostic["correct"] = (
    y_true_diag == y_pred_diag
)

# Percentuale di slot dell'encoding contenenti uno spike.
valid_diagnostic["spike_rate"] = (
    x_valid_diag.mean(axis=(1, 2))
)

# Misure sul segnale prima dell'encoding.
raw_float = x_valid_raw_diag.astype(np.float64)

valid_diagnostic["raw_rms"] = np.sqrt(
    np.mean(np.square(raw_float), axis=(1, 2))
)

valid_diagnostic["raw_std"] = np.std(
    raw_float,
    axis=(1, 2),
)

valid_diagnostic["raw_peak_to_peak"] = np.ptp(
    raw_float,
    axis=(1, 2),
)

# Punteggi prodotti dalla rete.
valid_diagnostic["rest_score"] = scores_diag[:, 0]

valid_diagnostic["true_score"] = [
    scores_diag[index, true_class]
    for index, true_class in enumerate(y_true_diag)
]

valid_diagnostic["pred_score"] = (
    scores_diag.max(axis=1)
)

valid_diagnostic["rest_minus_true"] = (
    valid_diagnostic["rest_score"]
    - valid_diagnostic["true_score"]
)

# Identificazione degli errori gesto -> rest.
gesture_to_rest_mask = (
    (valid_diagnostic["true"] != 0)
    & (valid_diagnostic["pred"] == 0)
)

valid_diagnostic["gesture_to_rest"] = (
    gesture_to_rest_mask
)

rest_errors = valid_diagnostic[
    gesture_to_rest_mask
].copy()

# ------------------------------------------------------------
# 4. Riepilogo generale
# ------------------------------------------------------------

gesture_windows = int(
    np.sum(y_true_diag != 0)
)

gesture_to_rest_count = int(
    gesture_to_rest_mask.sum()
)

print("\n" + "=" * 70)
print("DIAGNOSTICA GESTI CLASSIFICATI COME REST")
print("=" * 70)

print("Finestre validation:", len(valid_diagnostic))
print("Finestre gesto:", gesture_windows)
print(
    "Gesti classificati come rest:",
    gesture_to_rest_count,
)

if gesture_windows > 0:
    error_percentage = (
        100
        * gesture_to_rest_count
        / gesture_windows
    )

    print(
        "Percentuale gesto -> rest:",
        f"{error_percentage:.2f}%",
    )

print("\nErrori per classe vera:")

error_by_class = (
    rest_errors.groupby("true_name")
    .size()
    .reindex(
        ["sasso", "forbici", "carta"],
        fill_value=0,
    )
    .rename("gesture_to_rest")
)

display(error_by_class.to_frame())

# ------------------------------------------------------------
# 5. Errori per file e soggetto
# ------------------------------------------------------------

print("\nErrori gesto -> rest per file:")

errors_by_file = (
    rest_errors.groupby(["file", "true_name"])
    .size()
    .rename("errors")
    .reset_index()
    .sort_values(
        "errors",
        ascending=False,
    )
)

display(errors_by_file)

print("\nTasso di errore gesto -> rest per file:")

file_summary = (
    valid_diagnostic[
        valid_diagnostic["true"] != 0
    ]
    .groupby("file")
    .agg(
        gesture_windows=("true", "size"),
        gesture_to_rest=("gesture_to_rest", "sum"),
        mean_spike_rate=("spike_rate", "mean"),
        mean_raw_std=("raw_std", "mean"),
        mean_label_ratio=("label_ratio", "mean"),
    )
)

file_summary["error_rate_pct"] = (
    100
    * file_summary["gesture_to_rest"]
    / file_summary["gesture_windows"]
)

display(
    file_summary.sort_values(
        "error_rate_pct",
        ascending=False,
    )
)

# ------------------------------------------------------------
# 6. Confronto tra finestre corrette e sbagliate
# ------------------------------------------------------------

gesture_comparison = valid_diagnostic[
    valid_diagnostic["true"] != 0
].copy()

gesture_comparison["result"] = np.where(
    gesture_comparison["gesture_to_rest"],
    "classificata rest",
    np.where(
        gesture_comparison["correct"],
        "corretta",
        "altro errore",
    ),
)

print("\nConfronto attività corretta/sbagliata:")

comparison_table = (
    gesture_comparison[
        gesture_comparison["result"].isin([
            "corretta",
            "classificata rest",
        ])
    ]
    .groupby(["true_name", "result"])
    .agg(
        windows=("true", "size"),
        spike_rate_mean=("spike_rate", "mean"),
        spike_rate_std=("spike_rate", "std"),
        raw_std_mean=("raw_std", "mean"),
        raw_rms_mean=("raw_rms", "mean"),
        peak_to_peak_mean=("raw_peak_to_peak", "mean"),
        label_ratio_mean=("label_ratio", "mean"),
        rest_score_mean=("rest_score", "mean"),
        true_score_mean=("true_score", "mean"),
    )
)

display(comparison_table)

# ------------------------------------------------------------
# 7. Elenco delle finestre errate
# ------------------------------------------------------------

print(
    "\nFinestre gesto -> rest ordinate "
    "dal modello più incerto al più convinto:"
)

columns_to_show = [
    "file",
    "start_sample",
    "end_sample",
    "start_s",
    "end_s",
    "true_name",
    "label_ratio",
    "spike_rate",
    "raw_std",
    "raw_peak_to_peak",
    "rest_score",
    "true_score",
    "rest_minus_true",
]

display(
    rest_errors[columns_to_show]
    .sort_values("rest_minus_true")
    .head(50)
)

# ------------------------------------------------------------
# 8. Grafici riassuntivi
# ------------------------------------------------------------

plot_data = gesture_comparison[
    gesture_comparison["result"].isin([
        "corretta",
        "classificata rest",
    ])
].copy()

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5),
)

plot_data.boxplot(
    column="spike_rate",
    by="result",
    ax=axes[0],
)

axes[0].set_title("Spike rate")
axes[0].set_xlabel("")
axes[0].set_ylabel("Spike rate")

plot_data.boxplot(
    column="raw_std",
    by="result",
    ax=axes[1],
)

axes[1].set_title("Variabilità del segnale")
axes[1].set_xlabel("")
axes[1].set_ylabel("STD raw")

for class_id in (1, 2, 3):
    class_mask = (
        y_true_diag == class_id
    )

    axes[2].scatter(
        valid_diagnostic.loc[
            class_mask,
            "spike_rate",
        ],
        valid_diagnostic.loc[
            class_mask,
            "rest_minus_true",
        ],
        s=18,
        alpha=0.6,
        label=CLASS_NAMES[class_id],
    )

axes[2].axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

axes[2].set_title(
    "Decisione rest rispetto alla classe vera"
)
axes[2].set_xlabel("Spike rate")
axes[2].set_ylabel(
    "score rest − score classe vera"
)
axes[2].legend()

fig.suptitle("")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 9. Grafico errori per file
# ------------------------------------------------------------

if not errors_by_file.empty:
    pivot_errors = errors_by_file.pivot_table(
        index="file",
        columns="true_name",
        values="errors",
        fill_value=0,
    )

    pivot_errors.plot(
        kind="bar",
        stacked=True,
        figsize=(14, 5),
    )

    plt.title(
        "Finestre gesto classificate come rest per file"
    )
    plt.xlabel("File")
    plt.ylabel("Numero di errori")
    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Classe vera")
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 10. Visualizzazione di alcune finestre sbagliate
# ------------------------------------------------------------

error_indices = (
    rest_errors
    .sort_values(
        "rest_minus_true",
        ascending=False,
    )
    .head(MAX_WINDOWS_TO_PLOT)
    .index
    .to_numpy()
)

for window_index in error_indices:
    row = valid_diagnostic.loc[window_index]
    window = x_valid_raw_diag[window_index]

    channel_count = window.shape[1]

    fig, axes = plt.subplots(
        channel_count,
        1,
        figsize=(14, 10),
        sharex=True,
    )

    if channel_count == 1:
        axes = [axes]

    time_ms = (
        np.arange(window.shape[0])
        / FS_DIAGNOSTIC
        * 1000
    )

    for channel_index, axis in enumerate(axes):
        axis.plot(
            time_ms,
            window[:, channel_index],
            linewidth=0.8,
        )

        axis.set_ylabel(
            f"CH{channel_index + 1}"
        )

    axes[-1].set_xlabel(
        "Tempo nella finestra [ms]"
    )

    fig.suptitle(
        f"{row['file']} | "
        f"{row['start_s']:.3f}–"
        f"{row['end_s']:.3f} s | "
        f"true={row['true_name']} → rest | "
        f"spike rate={row['spike_rate']:.4f} | "
        f"rest−true={row['rest_minus_true']:.2f}"
    )

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 11. Confronto errore/corretta per ogni gesto
# ------------------------------------------------------------

for class_id in (1, 2, 3):
    wrong_indices = valid_diagnostic.index[
        (valid_diagnostic["true"] == class_id)
        & valid_diagnostic["gesture_to_rest"]
    ].to_numpy()

    correct_indices = valid_diagnostic.index[
        (valid_diagnostic["true"] == class_id)
        & valid_diagnostic["correct"]
    ].to_numpy()

    if (
        len(wrong_indices) == 0
        or len(correct_indices) == 0
    ):
        continue

    wrong_index = wrong_indices[0]
    correct_index = correct_indices[0]

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(14, 7),
        sharex=True,
    )

    axes[0].plot(
        x_valid_raw_diag[wrong_index]
    )
    axes[0].set_title(
        f"{CLASS_NAMES[class_id]} classificata REST | "
        f"spike rate="
        f"{valid_diagnostic.loc[wrong_index, 'spike_rate']:.4f}"
    )

    axes[1].plot(
        x_valid_raw_diag[correct_index]
    )
    axes[1].set_title(
        f"{CLASS_NAMES[class_id]} classificata correttamente | "
        f"spike rate="
        f"{valid_diagnostic.loc[correct_index, 'spike_rate']:.4f}"
    )

    axes[1].set_xlabel(
        "Campione nella finestra"
    )

    plt.tight_layout()
    plt.show()

print("\nDiagnostica completata.")
print(
    "DataFrame completo disponibile come: "
    "valid_diagnostic"
)
print(
    "Errori gesto -> rest disponibili come: "
    "rest_errors"
)

## 10. Valutazione su test set

Calcolo delle predizioni sul test set, confusion matrix e classification report.


In [ ]:
if RUN_TEST:
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
    
    model.eval()
    
    y_true = []
    y_pred = []
    scores = []
    
    with torch.no_grad():
        for input, label in test_dataloader:
            input = input.to(device)
    
            output = model(input)
    
            # output shape: (batch, num_classes, time)
            pred = slayer.classifier.Rate.predict(output)
    
            # score grezzo: spike totali per classe lungo il tempo
            batch_scores = output.sum(dim=2)
    
            y_true.extend(label.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())
            scores.append(batch_scores.cpu().numpy())
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    scores = np.concatenate(scores, axis=0)
    
    test_acc = (y_true == y_pred).mean()
    
    print(f"Test accuracy: {test_acc:.4f} ({test_acc * 100:.2f}%)")
    print("scores shape:", scores.shape)

else:
    print("Test set bloccato: valutazione saltata.")


In [ ]:
if RUN_TEST:
    labels = [0, 1, 2, 3]
    
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels
    )
    
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    plt.title("Confusion Matrix - Test")
    plt.show()

In [ ]:
if RUN_TEST:
    from sklearn.metrics import classification_report
    
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=["rest", "sasso", "forbici", "carta"],
        digits=4
    ))

## 11. Salvataggio risultati

Ogni run viene salvato in results/ con configurazione, metriche, predizioni, confusion matrix, history e modello.


In [ ]:
if RUN_TEST:
    from pathlib import Path
    from datetime import datetime
    from sklearn.metrics import (
        accuracy_score,
        f1_score,
        recall_score,
        precision_score,
        confusion_matrix,
        classification_report,
        ConfusionMatrixDisplay,
    )
    import json
    
    RESULTS_DIR = PROJECT_ROOT / "results"
    RESULTS_DIR.mkdir(exist_ok=True)
    
    run_name = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = RESULTS_DIR / run_name
    run_dir.mkdir(exist_ok=True)
    
    labels = [0, 1, 2, 3]
    class_names = ["rest", "sasso", "forbici", "carta"]
    
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "per_class_precision": {
            class_names[i]: float(v)
            for i, v in enumerate(
                precision_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
            )
        },
        "per_class_recall": {
            class_names[i]: float(v)
            for i, v in enumerate(
                recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
            )
        },
        "per_class_f1": {
            class_names[i]: float(v)
            for i, v in enumerate(
                f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
            )
        },
    }
    
    config = {
        "seed": int(SEED),
        "data_version": DATA_VERSION,
        "split_mode": SPLIT_MODE,
        "filename_split_patterns": FILENAME_SPLIT_PATTERNS,
        "split_ratios": SPLIT_RATIOS,
        "dataset_path": str(DATASET_PATH),
        "class_names": class_names,
        "augmentation": {
            "enabled": bool(USE_TRAIN_AUGMENTATION),
            "copies": int(AUGMENTATION_COPIES),
            "global_gain_range": list(GLOBAL_GAIN_RANGE),
            "channel_gain_range": list(CHANNEL_GAIN_RANGE),
            "noise_std_fraction_range": list(NOISE_STD_FRACTION_RANGE),
            "crosstalk_range": list(CROSSTALK_RANGE),
            "temporal_shift": False,
            "output_dtype": "int32",
        },
        "encoding_mode": ENCODING_MODE,
        "lambda_d": float(LAMBDA_D),
        "max_derivative_order": int(MAX_DERIVATIVE_ORDER),
        "delta_by_order": [float(x) for x in DELTA_BY_ORDER[:MAX_DERIVATIVE_ORDER + 1]],
        "window_size": int(WINDOW_SIZE),
        "window_shift": int(WINDOW_SHIFT),
        "min_label_ratio": float(MIN_LABEL_RATIO),
        "delta": float(DELTA),
        "batch_size": int(BATCH_SIZE),
        "epochs": int(EPOCHS),
        "patience": int(PATIENCE),
        "loss_type": LOSS_TYPE,
        "true_rate": float(TRUE_RATE),
        "false_rate": float(FALSE_RATE),
        "class_weights": CLASS_WEIGHTS.tolist() if LOSS_TYPE == "weighted_spike_rate" else None,
        "num_classes": int(NUM_CLASSES),
        "input_features": int(INPUT_FEATURES),
        "network": "Dense 64-128-4",
    }
    
    history = {
        "train_loss": [float(x) for x in train_loss_hist],
        "valid_loss": [float(x) for x in valid_loss_hist],
        "train_acc": [float(x) for x in train_acc_hist],
        "valid_acc": [float(x) for x in valid_acc_hist],
    }
    
    test_results_df = pd.DataFrame({
        "y_true": y_true,
        "y_pred": y_pred,
        "correct": y_true == y_pred,
    })
    
    if "scores" in globals():
        test_results_df["max_score"] = scores.max(axis=1)
        test_results_df["pred_score"] = scores[np.arange(len(y_pred)), y_pred]
    
        sorted_scores = np.sort(scores, axis=1)
        test_results_df["second_score"] = sorted_scores[:, -2]
        test_results_df["margin"] = test_results_df["max_score"] - test_results_df["second_score"]
    
        for i, class_name in enumerate(class_names):
            test_results_df[f"score_{class_name}"] = scores[:, i]
    
    report_text = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=class_names,
        digits=4,
        zero_division=0,
    )
    
    with open(run_dir / "config.json", "w") as f:
        json.dump(config, f, indent=4)
    
    with open(run_dir / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=4)
    
    with open(run_dir / "history.json", "w") as f:
        json.dump(history, f, indent=4)
    
    with open(run_dir / "classification_report.txt", "w") as f:
        f.write(report_text)
    
    test_results_df.to_csv(run_dir / "test_predictions.csv", index=False)
    
    np.save(run_dir / "confusion_matrix.npy", cm)
    np.save(run_dir / "y_true.npy", y_true)
    np.save(run_dir / "y_pred.npy", y_pred)
    
    if "scores" in globals():
        np.save(run_dir / "scores.npy", scores)
    
    torch.save(model.state_dict(), run_dir / "model_state_dict.pt")
    torch.save(model, run_dir / "model_best.pt")
    
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    plt.title("Confusion Matrix - Test")
    plt.savefig(run_dir / "confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.show()
    
    print("Risultati salvati in:", run_dir)
    print("\nMetrics:")
    print(json.dumps(metrics, indent=4))
    print("\nClassification report:")
    print(report_text)


## 12. Debug e visualizzazioni

Celle di ispezione utili per capire gli errori: spike rate per classe, finestre rest predette come gesto e plot del segnale completo.


In [ ]:
if RUN_TEST :
    def spike_rate_by_class(x, y, split_name):
        rows = []
    
        for cls in sorted(np.unique(y)):
            mask = y == cls
            x_cls = x[mask]
    
            spike_rate = x_cls.mean()
            spikes_per_window = x_cls.sum(axis=(1, 2))
    
            rows.append({
                "split": split_name,
                "class": cls,
                "n_windows": mask.sum(),
                "spike_rate": spike_rate,
                "spikes_per_window_mean": spikes_per_window.mean(),
                "spikes_per_window_std": spikes_per_window.std(),
                "spikes_per_window_min": spikes_per_window.min(),
                "spikes_per_window_max": spikes_per_window.max(),
            })
    
        return pd.DataFrame(rows)
    
    
    spike_class_stats = pd.concat([
        spike_rate_by_class(x_train, y_train, "train"),
        spike_rate_by_class(x_valid, y_valid, "valid"),
        spike_rate_by_class(x_test, y_test, "test"),
    ], ignore_index=True)
    
    display(spike_class_stats)

In [ ]:
if RUN_TEST:
    rest_mask = y_test == 0
    rest_spikes = x_test[rest_mask].sum(axis=(1, 2))
    
    gesture_mask = y_test != 0
    gesture_spikes = x_test[gesture_mask].sum(axis=(1, 2))
    
    print("Rest spike percentiles:")
    for p in [50, 75, 90, 95, 99, 100]:
        print(f"p{p}: {np.percentile(rest_spikes, p):.1f}")
    
    print("\nGesture spike percentiles:")
    for p in [0, 5, 10, 25, 50]:
        print(f"p{p}: {np.percentile(gesture_spikes, p):.1f}")

In [ ]:
if RUN_TEST:
    rest_indices = np.where(y_true == 0)[0]
    rest_pred = y_pred[rest_indices]
    rest_spikes_test = x_test[rest_indices].sum(axis=(1, 2))
    
    correct_rest = rest_pred == 0
    wrong_rest = rest_pred != 0
    
    print("Rest corretto - spike medi:", rest_spikes_test[correct_rest].mean())
    print("Rest sbagliato - spike medi:", rest_spikes_test[wrong_rest].mean())
    
    print("Rest corretto - mediana:", np.median(rest_spikes_test[correct_rest]))
    print("Rest sbagliato - mediana:", np.median(rest_spikes_test[wrong_rest]))

In [ ]:
if RUN_TEST:
    # Indici delle finestre test che sono rest ma predette come gesto
    false_gesture_idx = np.where((y_true == 0) & (y_pred != 0))[0]
    
    print("False gesture da rest:", len(false_gesture_idx))
    print("Prime predizioni:", y_pred[false_gesture_idx[:20]])
    
    def plot_test_window(idx, channels=None):
        if channels is None:
            channels = list(range(len(CHANNEL_COLS)))
    
        raw = x_test_raw[idx]     # shape: (time, channels)
        spikes = x_test[idx]      # shape: (channels*2, time)
    
        true_label = y_true[idx]
        pred_label = y_pred[idx]
    
        t = np.arange(raw.shape[0])
    
        fig, axes = plt.subplots(len(channels), 1, figsize=(16, 2.2 * len(channels)), sharex=True)
    
        if len(channels) == 1:
            axes = [axes]
    
        for ax, ch in zip(axes, channels):
            signal = raw[:, ch]
    
            pos = spikes[ch * 2, :] == 1
            neg = spikes[ch * 2 + 1, :] == 1
    
            ax.plot(t, signal, color="black", linewidth=1)
            ax.scatter(t[pos], signal[pos], color="red", s=18, label="pos spike")
            ax.scatter(t[neg], signal[neg], color="blue", s=18, label="neg spike")
    
            ax.set_ylabel(CHANNEL_COLS[ch])
            ax.grid(True, alpha=0.25)
    
        axes[0].set_title(
            f"Test window {idx} | true={true_label} | pred={pred_label} | "
            f"total spikes={spikes.sum():.0f}"
        )
    
        axes[-1].set_xlabel("Time step")
        plt.tight_layout()
        plt.show()

In [ ]:
if RUN_TEST:
    for idx in false_gesture_idx[:5]:
        plot_test_window(idx, channels=[0, 1, 2, 3, 4, 5, 6, 7])

### Plot del file completo

Visualizzazione di un file intero con tutti gli 8 canali e gli spike generati dal delta encoding.


In [ ]:
FILE_NAME = list(all_data.keys())[9]   # cambia qui
START = None                           # es. 0
END = None                             # es. 5000

emg_full = all_data[FILE_NAME]["emg"]
label_full = all_data[FILE_NAME]["label"]

if START is not None or END is not None:
    emg_full = emg_full[START:END]
    label_full = label_full[START:END]

def delta_encode_full_signal(emg, delta):
    time_steps, channels = emg.shape
    spikes = np.zeros((channels * 2, time_steps), dtype=np.float32)

    baseline = emg[0].copy()

    for t in range(time_steps):
        diff = emg[t] - baseline

        pos = diff > delta
        neg = diff < -delta

        spikes[0::2, t] = pos.astype(np.float32)
        spikes[1::2, t] = neg.astype(np.float32)

        baseline[pos | neg] = emg[t, pos | neg]

    return spikes

spikes_full = delta_encode_full_signal(emg_full, DELTA)

t = np.arange(len(emg_full))

fig, axes = plt.subplots(8, 1, figsize=(22, 16), sharex=True)

for ch in range(8):
    ax = axes[ch]

    signal = emg_full[:, ch]
    pos = spikes_full[ch * 2, :] == 1
    neg = spikes_full[ch * 2 + 1, :] == 1

    ax.plot(t, signal, color="black", linewidth=0.8)

    ax.scatter(
        t[pos],
        signal[pos],
        color="red",
        s=12,
        label="pos spike" if ch == 0 else None,
        zorder=3,
    )

    ax.scatter(
        t[neg],
        signal[neg],
        color="blue",
        s=12,
        label="neg spike" if ch == 0 else None,
        zorder=3,
    )

    ax.set_ylabel(CHANNEL_COLS[ch])
    ax.grid(True, alpha=0.25)

axes[0].set_title(f"File intero: {FILE_NAME} | DELTA={DELTA}")
axes[-1].set_xlabel("Campione")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")

plt.tight_layout()
plt.show()

print("Spike totali:", spikes_full.sum())
for ch in range(8):
    pos_count = spikes_full[ch * 2].sum()
    neg_count = spikes_full[ch * 2 + 1].sum()
    print(f"{CHANNEL_COLS[ch]} | pos: {pos_count:.0f} | neg: {neg_count:.0f} | total: {pos_count + neg_count:.0f}")

### Plot del file completo con label

Versione del plot che aggiunge in basso la traccia della label nel tempo.


In [ ]:
FILE_NAME = next((name for name in all_data if "sasso" in name.lower()), next(iter(all_data)))
START = None                             # es. 0
END = None                               # es. 10000

emg_full = all_data[FILE_NAME]["emg"]
label_full = all_data[FILE_NAME]["label"]

if START is not None or END is not None:
    emg_full = emg_full[START:END]
    label_full = label_full[START:END]

def delta_encode_full_signal(emg, delta):
    time_steps, channels = emg.shape
    spikes = np.zeros((channels * 2, time_steps), dtype=np.float32)

    baseline = emg[0].copy()

    for t in range(time_steps):
        diff = emg[t] - baseline

        pos = diff > delta
        neg = diff < -delta

        spikes[0::2, t] = pos.astype(np.float32)
        spikes[1::2, t] = neg.astype(np.float32)

        baseline[pos | neg] = emg[t, pos | neg]

    return spikes

spikes_full = delta_encode_full_signal(emg_full, DELTA)

t = np.arange(len(emg_full))

fig, axes = plt.subplots(9, 1, figsize=(22, 18), sharex=True)

for ch in range(8):
    ax = axes[ch]

    signal = emg_full[:, ch]
    pos = spikes_full[ch * 2, :] == 1
    neg = spikes_full[ch * 2 + 1, :] == 1

    ax.plot(t, signal, color="black", linewidth=0.8)

    ax.scatter(
        t[pos],
        signal[pos],
        color="red",
        s=10,
        label="pos spike" if ch == 0 else None,
        zorder=3,
    )

    ax.scatter(
        t[neg],
        signal[neg],
        color="blue",
        s=10,
        label="neg spike" if ch == 0 else None,
        zorder=3,
    )

    ax.set_ylabel(CHANNEL_COLS[ch])
    ax.grid(True, alpha=0.25)

# Label sotto ai canali
ax_label = axes[8]
ax_label.plot(t, label_full, color="green", linewidth=1.2)
ax_label.set_ylabel("label")
ax_label.set_xlabel("Campione")
ax_label.grid(True, alpha=0.25)

axes[0].set_title(f"File intero: {FILE_NAME} | DELTA={DELTA}")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right")

plt.tight_layout()
plt.show()

print("Spike totali:", spikes_full.sum())
for ch in range(8):
    pos_count = spikes_full[ch * 2].sum()
    neg_count = spikes_full[ch * 2 + 1].sum()
    print(
        f"{CHANNEL_COLS[ch]} | "
        f"pos: {pos_count:.0f} | "
        f"neg: {neg_count:.0f} | "
        f"total: {pos_count + neg_count:.0f}"
    )

### Confronto Delta Encoding

#### Funzioni encoder

In [ ]:
def delta_encode_standard(signal, delta):
    """
    Delta encoding standard:
    - spike positivo se x supera il riferimento di +delta
    - spike negativo se x scende sotto il riferimento di -delta
    - quando spika, il riferimento viene portato al valore corrente
    """
    signal = np.asarray(signal, dtype=np.float32)
    n_samples, n_channels = signal.shape

    spikes = np.zeros((n_samples, n_channels * 2), dtype=np.float32)
    ref = signal[0].copy()

    for t in range(1, n_samples):
        diff = signal[t] - ref

        pos = diff >= delta
        neg = diff <= -delta

        spikes[t, 0::2] = pos.astype(np.float32)
        spikes[t, 1::2] = neg.astype(np.float32)

        fired = pos | neg
        ref[fired] = signal[t, fired]

    return spikes


def delta_encode_lambda(signal, delta, lambda_d=0.001):
    """
    Delta encoding con lambda:
    - il riferimento decade lentamente verso 0
    - se x supera ref di +/- delta, genera spike
    - dopo lo spike, il riferimento si muove di +/- delta, non salta direttamente a x
    """
    signal = np.asarray(signal, dtype=np.float32)
    n_samples, n_channels = signal.shape

    spikes = np.zeros((n_samples, n_channels * 2), dtype=np.float32)
    ref = signal[0].copy()

    for t in range(1, n_samples):
        ref = ref * (1.0 - lambda_d)

        diff = signal[t] - ref

        pos = diff >= delta
        neg = diff <= -delta

        for ch in range(n_channels):
            if pos[ch]:
                spikes[t, 2 * ch] = 1.0
                ref[ch] = ref[ch] + delta

            if neg[ch]:
                spikes[t, 2 * ch + 1] = 1.0
                ref[ch] = ref[ch] - delta

    return spikes


In [ ]:
RAW_DIR = Path("1ks_labeled")
FILTERED_DIR = Path("dataset")

FILES = {
    "carta": {
        "raw": RAW_DIR / "carta_1k_labeled.csv",
        "filtered": FILTERED_DIR / "carta_1k_filtered_labeled.csv",
    },
    "forbici": {
        "raw": RAW_DIR / "forbici_1k_labeled.csv",
        "filtered": FILTERED_DIR / "forbici_1k_filtered_labeled.csv",
    },
    "sasso": {
        "raw": RAW_DIR / "sasso_1k_labeled.csv",
        "filtered": FILTERED_DIR / "sasso_1k_filtered_labeled.csv",
    },
}

CHANNELS = [f"CH{i}" for i in range(1, 9)]

data = {}

for gesture, paths in FILES.items():
    data[gesture] = {}

    for version, path in paths.items():
        df = pd.read_csv(path)

        data[gesture][version] = {
            "x": df[CHANNELS].to_numpy(dtype=np.float32),
            "y": df["label"].to_numpy(),
            "df": df,
        }

        print(f"{gesture} | {version} | shape: {df.shape} | labels: {df['label'].value_counts().sort_index().to_dict()}")

In [ ]:
DELTAS = [100, 200, 300, 500, 600, 800, 1000]
LAMBDA_VALUES = [0.001, 0.005, 0.01, 0.02]

rows = []

for gesture, versions in data.items():
    for version, item in versions.items():
        x = item["x"]
        y = item["y"]

        for delta in DELTAS:
            spikes_std = delta_encode_standard(x, delta=delta)

            rows.append({
                "gesture_file": gesture,
                "data_version": version,
                "encoder": "standard",
                "delta": delta,
                "lambda_d": None,
                "total_spikes": int(spikes_std.sum()),
                "spike_rate": float(spikes_std.mean()),
                "rest_spike_rate": float(spikes_std[y == 0].mean()),
                "gesture_spike_rate": float(spikes_std[y != 0].mean()),
            })

            for lambda_d in LAMBDA_VALUES:
                spikes_lambda = delta_encode_lambda(x, delta=delta, lambda_d=lambda_d)

                rows.append({
                    "gesture_file": gesture,
                    "data_version": version,
                    "encoder": "lambda",
                    "delta": delta,
                    "lambda_d": lambda_d,
                    "total_spikes": int(spikes_lambda.sum()),
                    "spike_rate": float(spikes_lambda.mean()),
                    "rest_spike_rate": float(spikes_lambda[y == 0].mean()),
                    "gesture_spike_rate": float(spikes_lambda[y != 0].mean()),
                })

encoding_comparison = pd.DataFrame(rows)
encoding_comparison


In [ ]:
encoding_comparison["gesture_to_rest_ratio"] = (
    encoding_comparison["gesture_spike_rate"] / 
    (encoding_comparison["rest_spike_rate"] + 1e-9)
)

encoding_comparison.sort_values(
    ["gesture_to_rest_ratio", "gesture_spike_rate"],
    ascending=[False, False]
).head(30)

In [ ]:
plt.figure(figsize=(14, 6))

summary = encoding_comparison[
    (encoding_comparison["encoder"] == "standard")
].copy()

for version in ["raw", "filtered"]:
    subset = summary[summary["data_version"] == version]
    grouped = subset.groupby("delta")["spike_rate"].mean()

    plt.plot(grouped.index, grouped.values, marker="o", label=version)

plt.xlabel("Delta")
plt.ylabel("Spike rate medio")
plt.title("Standard delta encoding - raw vs filtered")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))

plot_df = encoding_comparison.copy()
plot_df["encoder_label"] = plot_df.apply(
    lambda r: "standard" if r["encoder"] == "standard" else f"lambda={r['lambda_d']}",
    axis=1,
)

for enc_label in plot_df["encoder_label"].unique():
    subset = plot_df[
        (plot_df["data_version"] == "raw") &
        (plot_df["encoder_label"] == enc_label)
    ]

    grouped = subset.groupby("delta")["spike_rate"].mean()
    plt.plot(grouped.index, grouped.values, marker="o", label=enc_label)

plt.xlabel("Delta")
plt.ylabel("Spike rate medio")
plt.title("Raw data - confronto standard vs lambda")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
out_dir = PROJECT_ROOT / "encoding_comparison_results"
out_dir.mkdir(exist_ok=True)

encoding_comparison.to_csv(out_dir / "encoding_comparison.csv", index=False)

print(f"Salvato in: {out_dir / 'encoding_comparison.csv'}")

In [ ]:
def plot_encoding_modes(
    gesture="sasso",
    delta=500,
    lambda_d=0.001,
    max_samples=None,
    channels=None,
):
    if channels is None:
        channels = list(range(8))

    modes = [
        ("raw", "standard"),
        ("raw", "lambda"),
        ("filtered", "standard"),
        ("filtered", "lambda"),
    ]

    fig, axes = plt.subplots(
        len(modes),
        len(channels),
        figsize=(4 * len(channels), 3 * len(modes)),
        sharex=False,
    )

    if len(channels) == 1:
        axes = axes.reshape(len(modes), 1)

    for row, (data_version, encoder_name) in enumerate(modes):
        x = data[gesture][data_version]["x"]

        if max_samples is not None:
            x_plot = x[:max_samples]
        else:
            x_plot = x

        if encoder_name == "standard":
            spikes = delta_encode_standard(x_plot, delta=delta)
        else:
            spikes = delta_encode_lambda(x_plot, delta=delta, lambda_d=lambda_d)

        t = np.arange(len(x_plot))

        for col, ch in enumerate(channels):
            ax = axes[row, col]

            signal = x_plot[:, ch]
            pos_spikes = spikes[:, ch * 2] > 0
            neg_spikes = spikes[:, ch * 2 + 1] > 0

            ax.plot(t, signal, color="black", linewidth=0.7)
            ax.scatter(t[pos_spikes], signal[pos_spikes], color="red", s=8)
            ax.scatter(t[neg_spikes], signal[neg_spikes], color="blue", s=8)
            ax.set_title(f"{data_version} | {encoder_name} | CH{ch + 1}")
            ax.grid(True, alpha=0.3)

            if col == 0:
                ax.set_ylabel("Ampiezza")

    fig.suptitle(
        f"{gesture} | delta={delta} | lambda_d={lambda_d}",
        fontsize=16,
    )

    plt.tight_layout()
    plt.show()


In [ ]:
plot_encoding_modes(
    gesture="sasso",
    delta=500,
    lambda_d=0.001,
    max_samples=5000,
    channels=[0, 1, 2, 3, 4, 5, 6, 7],
)


In [ ]:
plot_encoding_modes(
    gesture="sasso",
    delta=500,
    lambda_d=0.001,
    max_samples=None,
    channels=[0, 1, 2, 3, 4, 5, 6, 7],
)


In [ ]:
def plot_raw_vs_filtered_standard_encoding(
    gesture="sasso",
    channel=0,
    delta=500,
):
    raw_x = data[gesture]["raw"]["x"]
    filtered_x = data[gesture]["filtered"]["x"]

    raw_spikes = delta_encode_standard(raw_x, delta=delta)
    filtered_spikes = delta_encode_standard(filtered_x, delta=delta)

    t_raw = np.arange(len(raw_x))
    t_filtered = np.arange(len(filtered_x))

    raw_signal = raw_x[:, channel]
    filtered_signal = filtered_x[:, channel]

    raw_pos = raw_spikes[:, channel * 2] > 0
    raw_neg = raw_spikes[:, channel * 2 + 1] > 0

    filtered_pos = filtered_spikes[:, channel * 2] > 0
    filtered_neg = filtered_spikes[:, channel * 2 + 1] > 0

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(22, 8),
        sharex=True,
    )

    axes[0].plot(t_raw, raw_signal, color="black", linewidth=0.6)
    axes[0].scatter(
        t_raw[raw_pos],
        raw_signal[raw_pos],
        color="red",
        s=8,
        label="pos spike",
    )
    axes[0].scatter(
        t_raw[raw_neg],
        raw_signal[raw_neg],
        color="blue",
        s=8,
        label="neg spike",
    )
    axes[0].set_title(f"RAW | {gesture} | CH{channel + 1} | delta={delta}")
    axes[0].set_ylabel("Ampiezza")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(loc="upper right")

    axes[1].plot(t_filtered, filtered_signal, color="black", linewidth=0.6)
    axes[1].scatter(
        t_filtered[filtered_pos],
        filtered_signal[filtered_pos],
        color="red",
        s=8,
        label="pos spike",
    )
    axes[1].scatter(
        t_filtered[filtered_neg],
        filtered_signal[filtered_neg],
        color="blue",
        s=8,
        label="neg spike",
    )
    axes[1].set_title(f"FILTERED | {gesture} | CH{channel + 1} | delta={delta}")
    axes[1].set_xlabel("Campione")
    axes[1].set_ylabel("Ampiezza")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc="upper right")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_raw_vs_filtered_standard_encoding(
    gesture="sasso",
    channel=7,
    delta=500,
)

In [ ]:
def plot_raw_lambda_vs_filtered_standard_points(
    gesture="sasso",
    channel=0,
    delta=500,
    lambda_d=0.001,
):
    raw_x = data[gesture]["raw"]["x"]
    filtered_x = data[gesture]["filtered"]["x"]

    raw_spikes = delta_encode_lambda(raw_x, delta=delta, lambda_d=lambda_d)
    filtered_spikes = delta_encode_standard(filtered_x, delta=delta)

    t_raw = np.arange(len(raw_x))
    t_filtered = np.arange(len(filtered_x))

    raw_signal = raw_x[:, channel]
    filtered_signal = filtered_x[:, channel]

    raw_pos = raw_spikes[:, channel * 2] > 0
    raw_neg = raw_spikes[:, channel * 2 + 1] > 0

    filtered_pos = filtered_spikes[:, channel * 2] > 0
    filtered_neg = filtered_spikes[:, channel * 2 + 1] > 0

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(40, 14),
        sharex=True,
    )

    axes[0].plot(t_raw, raw_signal, color="black", linewidth=0.6)

    ax0_spikes = axes[0].twinx()
    ax0_spikes.scatter(
        t_raw[raw_pos],
        np.ones(raw_pos.sum()),
        color="red",
        s=8,
        label="pos spike",
    )
    ax0_spikes.scatter(
        t_raw[raw_neg],
        -np.ones(raw_neg.sum()),
        color="blue",
        s=8,
        label="neg spike",
    )
    ax0_spikes.set_ylim(-1.2, 1.2)
    ax0_spikes.set_yticks([-1, 0, 1])
    ax0_spikes.set_ylabel("Spike")

    axes[0].set_title(
        f"RAW + lambda encoding | {gesture} | CH{channel + 1} | delta={delta} | lambda_d={lambda_d}"
    )
    axes[0].set_ylabel("Ampiezza")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(t_filtered, filtered_signal, color="black", linewidth=0.6)

    ax1_spikes = axes[1].twinx()
    ax1_spikes.scatter(
        t_filtered[filtered_pos],
        np.ones(filtered_pos.sum()),
        color="red",
        s=3,
        label="pos spike",
    )
    ax1_spikes.scatter(
        t_filtered[filtered_neg],
        -np.ones(filtered_neg.sum()),
        color="blue",
        s=3,
        label="neg spike",
    )
    ax1_spikes.set_ylim(-1.2, 1.2)
    ax1_spikes.set_yticks([-1, 0, 1])
    ax1_spikes.set_ylabel("Spike")

    axes[1].set_title(
        f"FILTERED + standard encoding | {gesture} | CH{channel + 1} | delta={delta}"
    )
    axes[1].set_xlabel("Campione")
    axes[1].set_ylabel("Ampiezza")
    axes[1].grid(True, alpha=0.3)

    handles0, labels0 = ax0_spikes.get_legend_handles_labels()
    fig.legend(handles0, labels0, loc="upper right")

    plt.tight_layout()
    plt.show()


def plot_raw_leaky_vs_filtered_standard_points(*args, **kwargs):
    # Alias temporaneo per non rompere vecchie celle: usa il nuovo nome lambda.
    if "leak" in kwargs:
        kwargs["lambda_d"] = kwargs.pop("leak")
    return plot_raw_lambda_vs_filtered_standard_points(*args, **kwargs)


In [ ]:
plot_raw_lambda_vs_filtered_standard_points(
    gesture="sasso",
    channel=0,
    delta=500,
    lambda_d=0.0001,
)


In [ ]:
def delta_encode_lambda_slow(signal, delta, lambda_d=0.0001):
    return delta_encode_lambda(signal, delta=delta, lambda_d=lambda_d)


In [ ]:
plot_raw_lambda_vs_filtered_standard_points(
    gesture="sasso",
    channel=1,
    delta=1500,
    lambda_d=0.0001,
)


In [ ]:
def plot_raw_standard_lambda_vs_filtered_standard_points(
    gesture="sasso",
    channel=0,
    delta=500,
    lambda_d=0.0001,
):
    raw_x = data[gesture]["raw"]["x"]
    filtered_x = data[gesture]["filtered"]["x"]

    raw_standard_spikes = delta_encode_standard(raw_x, delta=delta)
    raw_lambda_spikes = delta_encode_lambda(raw_x, delta=delta, lambda_d=lambda_d)
    filtered_standard_spikes = delta_encode_standard(filtered_x, delta=delta)

    t_raw = np.arange(len(raw_x))
    t_filtered = np.arange(len(filtered_x))

    raw_signal = raw_x[:, channel]
    filtered_signal = filtered_x[:, channel]

    raw_standard_pos = raw_standard_spikes[:, channel * 2] > 0
    raw_standard_neg = raw_standard_spikes[:, channel * 2 + 1] > 0

    raw_lambda_pos = raw_lambda_spikes[:, channel * 2] > 0
    raw_lambda_neg = raw_lambda_spikes[:, channel * 2 + 1] > 0

    filtered_standard_pos = filtered_standard_spikes[:, channel * 2] > 0
    filtered_standard_neg = filtered_standard_spikes[:, channel * 2 + 1] > 0

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(40, 18),
        sharex=True,
    )

    # RAW + standard
    axes[0].plot(t_raw, raw_signal, color="black", linewidth=0.6)

    ax0_spikes = axes[0].twinx()
    ax0_spikes.scatter(
        t_raw[raw_standard_pos],
        np.ones(raw_standard_pos.sum()),
        color="red",
        s=3,
        label="pos spike",
    )
    ax0_spikes.scatter(
        t_raw[raw_standard_neg],
        -np.ones(raw_standard_neg.sum()),
        color="blue",
        s=3,
        label="neg spike",
    )
    ax0_spikes.set_ylim(-1.2, 1.2)
    ax0_spikes.set_yticks([-1, 0, 1])
    ax0_spikes.set_ylabel("Spike")

    axes[0].set_title(
        f"RAW + standard encoding | {gesture} | CH{channel + 1} | delta={delta}"
    )
    axes[0].set_ylabel("Ampiezza")
    axes[0].grid(True, alpha=0.3)

    # RAW + lambda
    axes[1].plot(t_raw, raw_signal, color="black", linewidth=0.6)

    ax1_spikes = axes[1].twinx()
    ax1_spikes.scatter(
        t_raw[raw_lambda_pos],
        np.ones(raw_lambda_pos.sum()),
        color="red",
        s=3,
        label="pos spike",
    )
    ax1_spikes.scatter(
        t_raw[raw_lambda_neg],
        -np.ones(raw_lambda_neg.sum()),
        color="blue",
        s=3,
        label="neg spike",
    )
    ax1_spikes.set_ylim(-1.2, 1.2)
    ax1_spikes.set_yticks([-1, 0, 1])
    ax1_spikes.set_ylabel("Spike")

    axes[1].set_title(
        f"RAW + lambda encoding | {gesture} | CH{channel + 1} | delta={delta} | lambda={lambda_d}"
    )
    axes[1].set_ylabel("Ampiezza")
    axes[1].grid(True, alpha=0.3)

    # FILTERED + standard
    axes[2].plot(t_filtered, filtered_signal, color="black", linewidth=0.6)

    ax2_spikes = axes[2].twinx()
    ax2_spikes.scatter(
        t_filtered[filtered_standard_pos],
        np.ones(filtered_standard_pos.sum()),
        color="red",
        s=3,
        label="pos spike",
    )
    ax2_spikes.scatter(
        t_filtered[filtered_standard_neg],
        -np.ones(filtered_standard_neg.sum()),
        color="blue",
        s=3,
        label="neg spike",
    )
    ax2_spikes.set_ylim(-1.2, 1.2)
    ax2_spikes.set_yticks([-1, 0, 1])
    ax2_spikes.set_ylabel("Spike")

    axes[2].set_title(
        f"FILTERED + standard encoding | {gesture} | CH{channel + 1} | delta={delta}"
    )
    axes[2].set_xlabel("Campione")
    axes[2].set_ylabel("Ampiezza")
    axes[2].grid(True, alpha=0.3)

    handles, labels = ax0_spikes.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_raw_standard_lambda_vs_filtered_standard_points(
    gesture="sasso",
    channel=6,
    delta=500,
    lambda_d=0.0001,
)